# FinSight — Financial News Risk Intelligence System
## WMG9B7 Individual Assessment | Sourabha K Kallapur

---

## Problem Statement

Financial institutions process thousands of news articles daily to assess market risk and make time-sensitive investment decisions. Manual review is prohibitively slow and inconsistent, while keyword-based systems miss nuanced financial signals. FinSight automates this pipeline by classifying financial news by topic, scoring urgency from tabular metadata, and generating structured risk briefs using a fine-tuned DistilBERT model. The system detects market regime shifts through statistical drift monitoring, enabling proactive risk management at scale.

## Setup Instructions

Run the following before executing any other cell:

```bash
!pip install -e ..
```

All cells must be run **in order**.  
Dataset loads automatically via HuggingFace `datasets`.

## Expected Runtime

| Environment       | Duration    |
|-------------------|-------------|
| Colab GPU (T4)    | ~45 minutes |
| CPU only          | ~3 hours    |

## Model Artefacts

- If `artefacts/distilbert_finsight.pt` **exists**, the notebook loads it and skips training.  
- If it does **not** exist, the notebook trains DistilBERT from scratch (3 epochs, ~14k samples).

## Problem Framing and Motivation

### The Problem — Why Financial News Classification Matters

Financial institutions must process thousands of news articles daily to assess market risk, yet manual review is prohibitively slow for real-time decision-making. Misclassifying a market-moving event — such as a central bank policy change embedded in diplomatic language — can result in significant financial losses within minutes of publication. Automated, high-accuracy classification enables real-time alerting, portfolio rebalancing triggers, and compliance logging. The accuracy and latency of such a system directly translate to competitive advantage and quantifiable risk mitigation.

### Why Deep Learning Over Classical ML

Traditional TF-IDF approaches treat text as a bag of words, discarding word order and contextual relationships. Consider: *“Fed raises rates unexpectedly, markets fall”* and *“Markets raise the Fed’s unexpected rates fall”* — identical TF-IDF vectors, very different meanings. DistilBERT (Sanh et al., 2019), a knowledge-distilled variant of BERT (Devlin et al., 2019), learns contextual embeddings where the same token receives different representations depending on its neighbours. The empirical comparison in this notebook demonstrates that contextual understanding yields substantially higher classification accuracy, particularly for ambiguous financial headlines.

### System Overview

FinSight is a 5-layer production pipeline:

| Layer | Component | Purpose |
|-------|-----------|----------|
| 1 | **Ingestion** | Pydantic schema validation and metadata extraction |
| 2 | **Preprocessing** | HTML stripping, URL removal, Unicode normalisation |
| 3 | **Classification** | Dual-model: TF-IDF + LogReg (fast) and DistilBERT (accurate) |
| 4 | **Risk Scoring** | Urgency assessment from 7 tabular metadata features |
| 5 | **Monitoring** | PSI / KS / Chi-Square statistical drift detection |

In [ ]:
# Install package in editable mode so src/ imports work in Colab
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".."],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("Package installed successfully.")
else:
    print(result.stdout[-1000:])
    print(result.stderr[-1000:])
    raise RuntimeError("Package installation failed.")

# Ensure project root is on sys.path
from pathlib import Path

_project_root = str(Path("..").resolve())
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import confusion_matrix, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
LABEL_TO_INT = {"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

In [ ]:
ds = load_dataset("ag_news")
train_ds = ds["train"]
test_ds  = ds["test"]

print(f"Train size: {len(train_ds):,}")
print(f"Test size:  {len(test_ds):,}")

train_labels = train_ds["label"]
test_labels  = test_ds["label"]

train_counts = pd.Series(train_labels).value_counts().sort_index()
test_counts  = pd.Series(test_labels).value_counts().sort_index()

dist_df = pd.DataFrame({
    "Class":       LABEL_NAMES,
    "Train count": train_counts.values,
    "Train %":     (train_counts.values / len(train_labels) * 100).round(1),
    "Test count":  test_counts.values,
    "Test %":      (test_counts.values / len(test_labels) * 100).round(1),
})
print("\nClass distribution:")
print(dist_df.to_string(index=False))

# Plot 1: Class distribution bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(4)
w = 0.35
ax.bar(x - w / 2, train_counts.values, w, label="Train", color="steelblue")
ax.bar(x + w / 2, test_counts.values,  w, label="Test",  color="coral")
ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES)
ax.set_xlabel("Class")
ax.set_ylabel("Article count")
ax.set_title("AG News Class Distribution (Train vs Test)")
ax.legend()
plt.tight_layout()
plt.show()

# Plot 2: Article length by class (boxplot)
lengths_by_class = {name: [] for name in LABEL_NAMES}
for item in train_ds:
    lengths_by_class[LABEL_NAMES[item["label"]]].append(len(item["text"].split()))

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(
    [lengths_by_class[n] for n in LABEL_NAMES],
    labels=LABEL_NAMES,
    patch_artist=True,
    medianprops={"color": "red", "linewidth": 2},
)
ax.set_xlabel("Class")
ax.set_ylabel("Article length (words)")
ax.set_title("Article Length Distribution by Class (Train set)")
plt.tight_layout()
plt.show()

# 3 sample articles per class
print("\n--- Sample articles (truncated to 100 chars) ---")
for label_idx, label_name in enumerate(LABEL_NAMES):
    samples = [item["text"] for item in train_ds if item["label"] == label_idx][:3]
    print(f"\n[{label_name}]")
    for i, s in enumerate(samples, 1):
        print(f"  {i}. {s[:100]}...")

In [ ]:
from src.preprocessing.pipeline import TextCleaner

cleaner = TextCleaner()

raw_samples = [train_ds[i]["text"] for i in range(3)]
print("=== TextCleaner.clean() demonstration ===\n")
for i, raw in enumerate(raw_samples, 1):
    cleaned = cleaner.clean(raw)
    print(f"--- Article {i} ---")
    print(f"  BEFORE: {raw[:120]!r}")
    print(f"  AFTER:  {cleaned[:120]!r}")
    print()

N_PER_CLASS = 5_000
all_texts  = train_ds["text"]
all_labels = train_ds["label"]

idx_per_class: dict = {c: [] for c in range(4)}
for i, lbl in enumerate(all_labels):
    idx_per_class[lbl].append(i)

selected_idx = []
for c in range(4):
    selected_idx.extend(idx_per_class[c][:N_PER_CLASS])

print(f"Cleaning {len(selected_idx):,} articles...")
subset_texts  = [cleaner.clean(all_texts[i]) for i in selected_idx]
subset_labels = [all_labels[i] for i in selected_idx]

x_train, x_val, x_test, y_train, y_val, y_test = cleaner.create_splits(
    subset_texts, subset_labels,
    test_size=0.2, val_size=0.1, random_state=SEED,
)

print(f"\nSplit sizes: train={len(x_train):,} | val={len(x_val):,} | test={len(x_test):,}")

def _class_dist(labels: list) -> str:
    counts = pd.Series(labels).value_counts().sort_index()
    return " | ".join(f"{LABEL_NAMES[i]}: {counts.get(i, 0)}" for i in range(4))

print("\nClass distribution per split:")
print(f"  Train : {_class_dist(y_train)}")
print(f"  Val   : {_class_dist(y_val)}")
print(f"  Test  : {_class_dist(y_test)}")

In [ ]:
from src.models.baseline import BaselineClassifier
from src.models.urgency import UrgencyScorer
from src.ingestion.features import extract_features
from src.ingestion.schema import ArticleIn

ARTEFACTS = Path("../artefacts")
ARTEFACTS.mkdir(exist_ok=True)

print("Training TF-IDF + Logistic Regression baseline...")
t0 = time.time()
baseline_clf = BaselineClassifier()
baseline_clf.fit(x_train, y_train)
print(f"  Done in {time.time() - t0:.1f}s")

baseline_preds     = baseline_clf.predict(x_test)
baseline_pred_ints = [LABEL_TO_INT[p.label] for p in baseline_preds]

baseline_acc         = sum(p == t for p, t in zip(baseline_pred_ints, y_test)) / len(y_test)
baseline_macro_f1    = float(f1_score(y_test, baseline_pred_ints, average="macro"))
baseline_weighted_f1 = float(f1_score(y_test, baseline_pred_ints, average="weighted"))
per_class_f1_base    = f1_score(y_test, baseline_pred_ints, average=None, labels=[0, 1, 2, 3])

print(f"\nBaseline Results on test set ({len(x_test):,} samples):")
print(f"  Accuracy:    {baseline_acc:.4f}")
print(f"  Macro-F1:    {baseline_macro_f1:.4f}")
print(f"  Weighted-F1: {baseline_weighted_f1:.4f}")
print("\n  Per-class F1:")
for i, name in enumerate(LABEL_NAMES):
    print(f"    {name:<12}: {per_class_f1_base[i]:.4f}")

cm_base = confusion_matrix(y_test, baseline_pred_ints)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_base, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(LABEL_NAMES, rotation=45, ha="right")
ax.set_yticklabels(LABEL_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Baseline Confusion Matrix (TF-IDF + LogReg)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm_base[i, j]), ha="center", va="center",
                color="white" if cm_base[i, j] > cm_base.max() / 2 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

latencies_baseline = []
for txt in x_test[:200]:
    t_start = time.perf_counter()
    baseline_clf.predict_single(txt)
    latencies_baseline.append((time.perf_counter() - t_start) * 1000)

p50_baseline = float(np.percentile(latencies_baseline, 50))
p99_baseline = float(np.percentile(latencies_baseline, 99))
print(f"\nBaseline inference latency (200 samples):")
print(f"  p50: {p50_baseline:.2f} ms  |  p99: {p99_baseline:.2f} ms")

baseline_clf.save(str(ARTEFACTS / "baseline_pipeline.joblib"))
print("\nSaved: artefacts/baseline_pipeline.joblib")

# Train UrgencyScorer with synthetic urgency labels derived from text features
def _make_urgency_label(text: str) -> int:
    safe = text[:9990] if len(text) > 9990 else text
    safe = safe if len(safe) >= 10 else safe + (" " * (10 - len(safe)))
    feats = extract_features(ArticleIn(text=safe))
    score = (
        feats["exclamation_count"] * 2.0
        + feats["question_count"]
        + feats["digit_ratio"] * 10.0
        + (1.0 if feats["word_count"] > 120 else 0.0)
    )
    if score >= 4: return 3
    elif score >= 2: return 2
    elif score >= 1: return 1
    else: return 0

print("\nFitting UrgencyScorer on training set...")
urgency_train_x = [
    extract_features(ArticleIn(text=t if len(t) >= 10 else t + " pad"))
    for t in x_train
]
urgency_train_y = [_make_urgency_label(t) for t in x_train]
urgency_scorer = UrgencyScorer()
urgency_scorer.fit(urgency_train_x, urgency_train_y)
urgency_scorer.save(str(ARTEFACTS / "urgency_pipeline.joblib"))
print("  Done. Saved: artefacts/urgency_pipeline.joblib")